# 03 — Preprocessing & Feature Engineering (HUPA-UCM)

**Goal of this notebook.** Transform the per-patient Excel files in `data/data_hupa/Preprocessed/` into model-ready artefacts in `data/processed/`. The HUPA-UCM authors (Hidalgo et al., 2024) already aligned every modality to a strict 5-minute grid; this notebook narrates *their* preprocessing transparently and then layers the thesis-specific preprocessing on top.

**Output artefacts.**
* `data/processed/hupa_5min_timestep.parquet` — per-timestep feature table with split labels.
* `data/processed/hupa_5min_sequences.npz` — sliding-window tensors `X_dynamic`, `X_static`, `y`, and metadata.
* `data/processed/hupa_static_features.csv` — one row per patient (clinical + derived features).
* `outputs/models/scalers.json` — per-subject Z-score + log1p+global Z-score parameters fit on the training portion only.

**Reading guide.** Markdown cells explain *why* each step is applied. Code cells run the pipeline implemented in `src/preprocessing.py` and inspect the outputs.

## 1. Environment

Detects Colab vs. local execution, mounts Drive on Colab, and adds `src/` to the Python path so the notebook can import `preprocessing` and `config` directly.

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/MyDrive/glucose-thesis/'
except Exception:
    IN_COLAB = False
    BASE_PATH = os.path.abspath(os.path.join(os.getcwd(), '..'))

BASE_PATH = Path(BASE_PATH)
SRC_PATH = BASE_PATH / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print('IN_COLAB:', IN_COLAB)
print('BASE_PATH:', BASE_PATH)

In [ ]:
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=FutureWarning)

import config as C
import preprocessing as pp

np.random.seed(C.SEED)

print('LOOKBACK_STEPS:', C.LOOKBACK_STEPS, f'({C.LOOKBACK_STEPS * C.SAMPLING_STEP_MIN} min)')
print('HORIZON_MINUTES:', C.HORIZON_MINUTES)
print('N_TRAIN_CAP:', C.N_TRAIN_CAP)
print('SPLIT fractions: train/val/test =',
      C.SPLIT_TRAIN_FRAC, C.SPLIT_VAL_FRAC, C.SPLIT_TEST_FRAC)

## 2. Part A — How the released data was preprocessed (paraphrased from Article_data.md §4.2)

The 25 Excel files in `data/data_hupa/Preprocessed/` are **not raw sensor exports**. They were produced by the glUCModel tool of the Adaptive and Bioinspired Systems (ABSys) group at Universidad Complutense de Madrid. The thesis takes those files as the starting point but documents the upstream pipeline so that downstream choices (e.g. retaining sensor caps, flagging modality absence) are defensible.

| Channel | Native cadence | Operation applied by the authors | Gap policy |
|---|---|---|---|
| **Glucose** (FreeStyle Libre 2) | ~15-min sweeps | Rounded to the nearest 5-min mark → subsampled to a strict 15-min grid → **linear interpolation** to fill back into a 5-min grid (max 1 h gap) | Gaps > 1 h discarded at the file boundaries |
| **Bolus insulin** | event-based | Summed within each 5-min bin | Empty bins → 0 |
| **Basal insulin** | 1 record per long-acting injection | Each daily long-acting dose divided by 288 (= 24 h × 12) and spread evenly across the day; overlapping intervals summed | Bins with no basal recording → 0 |
| **Carbohydrate intake** | event-based, in grams | Summed within each 5-min bin → divided by 10 (1 *serving* = 10 g) | Empty bins → 0 |
| **Heart rate** (Fitbit Ionic) | irregular 1–5 min | Rounded to the 5-min grid → linear interpolation to fill missing bins | Interpolated, not zero-filled |
| **Calories** (Fitbit) | 1-min | Summed across every 5 contiguous 1-min readings | Empty bins → 0 |
| **Steps** (Fitbit) | 1-min | Summed across every 5 contiguous 1-min readings | Empty bins → 0 |

**File-boundary trimming.** The authors retained the longest contiguous stretch in which both CGM glucose **and** heart-rate values were observed. Records at the beginning or end where one of those two channels was missing were discarded. All other modalities were zero-filled inside that retained window.

**Implications carried forward into this notebook.**

1. *Imputation is not a thesis contribution.* The glucose channel arrives already linearly interpolated. The notebook does not claim novelty on missing-data handling; it documents what was inherited.
2. *Zero in `basal_rate`, `bolus_volume_delivered`, or `carb_input` is ambiguous.* It can mean *no event recorded that bin* or *the patient never wore that recording method*. The notebook therefore adds **modality-availability flags** in §3 before the model ever sees the value.
3. *Glucose values exactly `40 mg/dL` are sensor-censored.* FreeStyle Libre 2 reports `LO` for any reading at or below 40 and `HI` for any reading above 400. The numbers in the released file are the post-censoring representation, not measurements. Two binary flags are added in §3 so the model can either down-weight them or be evaluated separately on censored windows.
4. *The released `time` column is monotonic and on a strict 5-min grid.* This was verified during Step 1 (data understanding); no resampling or de-duplication is needed here.

## 3. Part B — Thesis-specific preprocessing built on top

Every step below is implemented as a small pure function in `src/preprocessing.py`, then composed by `pp.run_preprocessing(...)`. Each step is paired with the evidence that motivates it.

### 3.1 Sensor-cap censoring flags
* `glucose_low_cap = 1` when `glucose <= 40 mg/dL` (FreeStyle Libre 2 `LO` representation).
* `glucose_high_extreme = 1` when `glucose > 400 mg/dL` (Libre `HI` representation).
* *Evidence:* §4.2 of Article_data.md identifies the device; the cohort summary in `data/interim/hupa_cohort_summary.csv` shows 0.38 % of rows globally are at the LO cap, peaking at 5.82 % for HUPA0002P. Values are retained, not removed, so the model can learn typical post-low recovery dynamics.

### 3.2 Treatment override for HUPA0011P (Pitfall #8)
* The static metadata lists HUPA0011P as CSII, yet the basal channel contains zero records for that patient. CSII pumps deliver continuous basal by definition, so leaving the label as `CSII` would teach the model the spurious rule *“CSII ⇒ no basal signal”*.
* The override changes `treatment` to `MDI` for HUPA0011P **only in the static feature table**; the raw modality flags still report `basal_available = 0`, which is the truthful description for that patient.

### 3.3 Modality-availability flags
* Per-patient binary flags: `basal_available`, `bolus_available`, `carb_available` (1 iff the patient has any non-zero recording across their whole timeline).
* Per-timestep rolling: `basal_coverage_24h` = fraction of the last 288 bins (24 h) in which `basal_rate > 0`. This distinguishes patients with continuous basal (≈ 1.0) from MDI patients with two daily injections (≈ 0.0–0.05) and from the four partial-coverage patients (HUPA0024/0026/0027/0028, 40–66 %).
* *Why both shapes?* Static flags identify *the patient*; the rolling fraction identifies *the local recording state*, which matters for windows that straddle pump removal or sensor down-time.

### 3.4 Time, glucose-derived, and event rolling features
* **Time:** `hour_sin`, `hour_cos`, `dayofweek_sin`, `dayofweek_cos` — cyclical encodings. Motivated by the circadian glucose peak around 07:00–10:00 documented in EDA §4.5.
* **Glucose-derived:** `glucose_velocity` (mg/dL · min⁻¹), `glucose_acceleration`, rolling means `glucose_30m_mean`, `glucose_60m_mean`, `glucose_120m_mean`. Justified by ACF half-life of 130 min in EDA §4.4.
* **Rolling sums on event streams:** `bolus_30m/60m/180m_sum`, `carb_60m/180m_sum`, `steps_30m/150m_sum`, `calories_30m_sum`, `heart_rate_30m_mean`. The 180-min bolus and carb spans capture the full pharmacokinetic / absorption window observed in EDA §4.6 peri-event traces.

### 3.5 Targets and chronological split
* Targets `target_30m`, `target_60m`, `target_90m` are computed by negative shifts of `glucose` within each patient. A patient's last 18 rows therefore have NaN targets and produce no training windows.
* The split is **strictly chronological inside each patient**: the first 70 % of their timeline → train, next 15 % → val, last 15 % → test.
* A **buffer of 18 rows** (the longest horizon) is removed at each split boundary. Without the buffer, a training row whose target falls at index `t + 18` could leak the first row of the validation partition.
* *Why not Leave-Patients-Out?* The thesis goal is short-term forecasting *for a known patient*. Per-patient chronological split is the protocol that matches that goal; an LOPO ablation will be reported separately in §8 once baselines exist.

### 3.6 Scaling
* **Per-subject Z-score** for continuous, slow-varying signals (`glucose`, `heart_rate`, `calories`, the four glucose-derived rolling means, velocity, acceleration). Justified by the strong inter-patient variance in glucose mean (113–201 mg/dL) and standard deviation (35–85 mg/dL) reported in EDA §4.7.
* **log1p + global Z-score** for sparse, heavy-tailed event sums (`basal_rate`, `bolus_volume_delivered`, `carb_input`, `steps`, and their rolling versions). The log1p compresses the heavy tail; the *global* Z-score is justified because the scale of bolus/carb/step values is comparable across patients of the same therapy class.
* **All scaler parameters are fit on `split == 'train'` rows only.** Validation and test rows are transformed using the same parameters. This is the operational guarantee against leakage.

### 3.7 Sequence build with adaptive-stride training cap
* Lookback = **24 steps (120 min)** as the primary configuration; 36 steps (180 min) reserved for ablation.
* For each anchor index `t` in a patient's timeline, the window is the slice `[t - 23, t]` and the targets are `glucose[t + 6]`, `glucose[t + 12]`, `glucose[t + 18]`.
* Windows whose lookback range crosses the buffer partition are dropped — there is no other way to bridge train/val cleanly.
* **Training-set sequence cap.** Per-patient adaptive-stride sub-sampling: if a patient has more than `N_TRAIN_CAP = 5000` candidate training anchors, anchors are picked at a deterministic stride `floor(n_train / 5000)`, sampled uniformly across the patient's timeline. Patients with fewer anchors keep all of them.
* **Validation and test keep `stride = 1`.** Every valid anchor in those partitions becomes a sequence. Metrics will therefore be reported both pooled and patient-averaged in §8.
* *Why this strategy?* Truncating the three long patients (HUPA0026/0027/0028 = 74.93 % of all rows) to the cohort median of 14 days would discard 97.6 % of HUPA0027P, destroy the multi-month seasonality, and erase the COVID-lockdown context for the 2020–2022 patients. Sample-cap with adaptive stride keeps that temporal diversity while preventing the long patients from dominating gradient updates.

### 3.8 Static feature table
* Clinical: `hba1c_pct`, `age_years`, `dx_time_years`, `weight_kg`, `height_cm`, `bmi` (computed) + one-hot `gender`, `treatment`.
* Derived (from each patient's *train* portion only): mean / std of glucose, hypo / TIR / hyper proportions, bolus and carb events per day, fraction of bins above the activity threshold, mean daily steps, mean heart rate, total duration, basal-recording fraction.
* Modality flags (binary): `basal_available`, `bolus_available`, `carb_available`.
* Numeric columns are Z-scored across the 25 patients — fitting on the full cohort is leakage-safe because each patient contributes exactly one row.

## 4. Execute the pipeline

`pp.run_preprocessing(...)` runs steps 3.1 → 3.8 in order and persists the artefacts. Set `participants=[...]` for a debug pass on a subset; leave it as `None` for the full 25-patient build.

In [ ]:
import time

DEBUG_MODE = False
DEBUG_PARTICIPANTS = ['HUPA0001P', 'HUPA0011P', 'HUPA0028P']

t0 = time.time()
artefacts = pp.run_preprocessing(
    base_path=BASE_PATH,
    participants=DEBUG_PARTICIPANTS if DEBUG_MODE else None,
    save=True,
)
elapsed = time.time() - t0
print(f'Pipeline completed in {elapsed:.1f} s')

## 5. Inspect the outputs

These cells verify that the artefacts on disk have sensible shapes, no missing values in the feature tensor, and that the long-patient cap was applied to the training set only.

In [ ]:
ts = artefacts['timestep_df']
bundle = artefacts['bundle']
static = artefacts['static_table']
boundaries = artefacts['boundaries']

print('Per-timestep table:')
print(f'  shape: {ts.shape}')
print(f'  split row counts:')
print(ts.groupby('split').size().to_string())

print('\nSequence bundle:')
print(f'  X_dynamic: {bundle.X_dynamic.shape}  (~{bundle.X_dynamic.nbytes / 1e6:.1f} MB float32)')
print(f'  X_static:  {bundle.X_static.shape}')
print(f'  y:         {bundle.y.shape}')
print(f'  NaN in X_dynamic? {bool(np.isnan(bundle.X_dynamic).any())}')
print(f'  NaN in y?         {bool(np.isnan(bundle.y).any())}')
print(f'  sequences by split:')
for s in ('train', 'val', 'test'):
    print(f'    {s}: {(bundle.split == s).sum():>6d}')

In [ ]:
# Per-patient sequence counts. The three long patients (HUPA0026/27/28) should
# each have n_sequences_train == N_TRAIN_CAP, while val/test stay proportional
# to their timeline length.
summary_path = BASE_PATH / 'outputs' / 'tables' / 'hupa_preprocessing_summary.csv'
summary = pd.read_csv(summary_path)
summary_view = summary[[
    'participant_id', 'n_rows_total',
    'n_sequences_train', 'n_sequences_val', 'n_sequences_test',
    'basal_available', 'bolus_available', 'carb_available'
]]
summary_view

In [ ]:
# Leakage sanity check: train, val, and test anchor timestamps for each
# patient must be strictly ordered (max(train) < min(val) and max(val) < min(test)).
anchor_time = pd.to_datetime(bundle.anchor_time)
leak_report = []
for pid in sorted(set(bundle.participant_ids)):
    mask = bundle.participant_ids == pid
    pid_split = bundle.split[mask]
    pid_time = anchor_time[mask]
    times = {s: pid_time[pid_split == s] for s in ('train', 'val', 'test')}
    if all(len(times[s]) > 0 for s in times):
        leak_report.append({
            'participant_id': pid,
            'train_max': times['train'].max(),
            'val_min': times['val'].min(),
            'val_max': times['val'].max(),
            'test_min': times['test'].min(),
            'train_max<val_min': times['train'].max() < times['val'].min(),
            'val_max<test_min': times['val'].max() < times['test'].min(),
        })
leak_df = pd.DataFrame(leak_report)
all_ok = leak_df['train_max<val_min'].all() and leak_df['val_max<test_min'].all()
print(f'No temporal leakage detected: {all_ok}')
leak_df.head()

In [ ]:
# Distribution sanity check: after per-subject Z-score, glucose in the train
# set should have mean ~ 0 and std ~ 1 *per patient*. We check the pooled
# train distribution as a coarse summary.
glu_idx = bundle.feature_names_dynamic.index('glucose')
train_glu = bundle.X_dynamic[bundle.split == 'train', :, glu_idx].ravel()
val_glu = bundle.X_dynamic[bundle.split == 'val', :, glu_idx].ravel()
test_glu = bundle.X_dynamic[bundle.split == 'test', :, glu_idx].ravel()
for name, arr in [('train', train_glu), ('val', val_glu), ('test', test_glu)]:
    print(f'{name} glucose (scaled): mean={arr.mean():+.3f}, std={arr.std():.3f}, n={len(arr):,}')

In [ ]:
# Visual: per-patient sequence count by split.
fig, ax = plt.subplots(figsize=(11, 5))
pids = sorted(summary['participant_id'].unique())
x = np.arange(len(pids))
ax.bar(x - 0.25, summary.set_index('participant_id').loc[pids, 'n_sequences_train'], width=0.25, label='train (capped)')
ax.bar(x,        summary.set_index('participant_id').loc[pids, 'n_sequences_val'],   width=0.25, label='val')
ax.bar(x + 0.25, summary.set_index('participant_id').loc[pids, 'n_sequences_test'],  width=0.25, label='test')
ax.axhline(C.N_TRAIN_CAP, color='black', linestyle='--', linewidth=0.8,
           label=f'N_TRAIN_CAP = {C.N_TRAIN_CAP}')
ax.set_xticks(x)
ax.set_xticklabels([p.replace('HUPA', '').replace('P', '') for p in pids], rotation=45)
ax.set_ylabel('# sequences')
ax.set_title('Per-patient sequence counts (lookback=24, horizons=30/60/90 min)')
ax.legend(loc='upper left')
ax.set_yscale('log')
plt.tight_layout()
fig_path = BASE_PATH / 'outputs' / 'figures' / '03_preprocessing_sequences_per_patient.png'
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=130)
plt.show()
print(f'Saved figure to {fig_path}')

In [ ]:
# Scaler parameter snapshot: confirm we have per-subject mean/std for glucose
# and global log1p parameters for bolus.
scalers_path = BASE_PATH / 'outputs' / 'models' / 'scalers.json'
with open(scalers_path, 'r', encoding='utf-8') as fh:
    scalers = json.load(fh)

print('Dynamic scalers — per-subject features:')
for feat in scalers['dynamic']['per_subject']:
    print(f'  - {feat}: {len(scalers["dynamic"]["per_subject"][feat])} patients')
print('\nDynamic scalers — log1p + global features:')
for feat, params in scalers['dynamic']['global_log1p'].items():
    print(f'  - {feat}: mean={params["mean"]:.3f}, std={params["std"]:.3f}')
print('\nStatic numeric scaler keys:', list(scalers['static'].keys())[:5], '...')

## 6. Closing notes

* Every artefact above is regeneratable by re-running `pp.run_preprocessing(BASE_PATH)`.
* The sequence file `hupa_5min_sequences.npz` is the single input to Step 4 (baselines) and Step 5 (hybrid model). Loaders consume it with `np.load(..., allow_pickle=True)` and read `feature_names_dynamic` / `feature_names_static` for column references.
* The next notebook (`04_model_training.ipynb`) will train the baseline ladder (persistence → Ridge → tree ensemble → LSTM/GRU) and produce the comparative table for `report.md §8`.
* If at any point the cap, split, or scaler choices need revision, edit the constants in `src/config.py` and re-run; the pipeline is deterministic for a fixed `SEED`.